In [1]:
!pip -q install arxiv


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.7 MB/s eta 0:00:00


In [2]:
# --- Colab: mount Google Drive ---
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    COLAB = True
except Exception:
    COLAB = False

import arxiv
import csv
import os
import re
import sys
import time
import zipfile
from pathlib import Path
from typing import Iterable, Optional
import requests
from requests.adapters import HTTPAdapter, Retry
import tempfile

# ====== OUTPUT PATHS (Drive) ======
# Colab's Drive path:
BASE_DIR = Path("/content/drive/MyDrive") if COLAB else Path.cwd()
OUTPUT_DIR = BASE_DIR / "arxiv_dumps"      # Drive folder to hold outputs
ZIP_NAME = "cs_cl_papers.zip"              # ZIP inside that folder
# ===================================

MAX_RESULTS = 500       # None = no explicit cap for the search (still paged)
PAGE_SIZE = 1000         # Larger pages = fewer API calls
DELAY_SECONDS = 0.0      # 0.0 for fastest; bump to 0.5 to be polite
RESUME_SKIP = True
SORT = "submitted"       # or "updated"

def ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def safe_filename(s: str) -> str:
    return re.sub(r'[^a-zA-Z0-9._-]+', '_', s)

def _make_http_session():
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    adapter = HTTPAdapter(max_retries=retries, pool_connections=8, pool_maxsize=8)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update({"User-Agent": "cscl-downloader/1.0"})
    return session

def stream_results_cs_cl(max_results: Optional[int],
                         sort_by: arxiv.SortCriterion,
                         page_size: int,
                         delay_seconds: float) -> Iterable[arxiv.Result]:
    search = arxiv.Search(
        query="cat:cs.CL AND submittedDate:[20240901 TO 20250101]",
        max_results=max_results,
        sort_by=sort_by,
        sort_order=arxiv.SortOrder.Descending
    )
    client = arxiv.Client(
        page_size=page_size,
        delay_seconds=delay_seconds,
        num_retries=5
    )
    return client.results(search)

def download_into_zip_drive():
    ensure_dir(OUTPUT_DIR)
    zip_path = OUTPUT_DIR / ZIP_NAME
    meta_path = OUTPUT_DIR / "metadata.csv"

    write_header = not meta_path.exists()
    meta_f = open(meta_path, "a", newline="", encoding="utf-8")
    writer = csv.writer(meta_f)
    if write_header:
        writer.writerow([
            "entry_id", "short_id", "title", "authors", "primary_category",
            "all_categories", "published", "updated", "doi", "journal_ref", "pdf_filename_in_zip"
        ])

    sort_by = arxiv.SortCriterion.SubmittedDate if SORT == "submitted" else arxiv.SortCriterion.LastUpdatedDate
    session = _make_http_session()

    total, errors = 0, 0

    # Create or open the ZIP. Preload existing names for resume-skip speed.
    with zipfile.ZipFile(zip_path, "a", compression=zipfile.ZIP_DEFLATED) as zf:
        existing = set(zf.namelist())

        try:
            for r in stream_results_cs_cl(
                max_results=MAX_RESULTS,
                sort_by=sort_by,
                page_size=PAGE_SIZE,
                delay_seconds=DELAY_SECONDS
            ):
                total += 1

                # Use "shortid_title.pdf" to make names unique+readable
                base_title = safe_filename((r.title or "").strip())[:140]
                short_id = r.get_short_id()
                filename = f"{short_id}_{base_title or short_id}.pdf"

                # Resume skip if that arcname already exists
                if RESUME_SKIP and filename in existing:
                    print(f"[{total}] SKIP (in zip): {filename}")
                    writer.writerow([
                        r.entry_id, short_id, (r.title or "").strip(),
                        "; ".join(a.name for a in r.authors),
                        r.primary_category,
                        ",".join(sorted(r.categories)),
                        r.published.isoformat() if r.published else "",
                        r.updated.isoformat() if r.updated else "",
                        r.doi or "", r.journal_ref or "",
                        filename
                    ])
                    continue

                try:
                    pdf_url = getattr(r, "pdf_url", None)
                    if not pdf_url:
                        pdf_url = r.entry_id.replace("/abs/", "/pdf/") + ".pdf"

                    # Stream to temp file to avoid big RAM spikes
                    with session.get(pdf_url, stream=True, timeout=60) as resp:
                        resp.raise_for_status()
                        with tempfile.NamedTemporaryFile(delete=False) as tmpf:
                            for chunk in resp.iter_content(chunk_size=1024 * 256):
                                if chunk:
                                    tmpf.write(chunk)
                            tmp_name = tmpf.name

                    # Add to zip, then remove temp
                    zf.write(tmp_name, arcname=filename)
                    os.remove(tmp_name)
                    existing.add(filename)

                    print(f"[{total}] OK  : {filename}")
                    writer.writerow([
                        r.entry_id, short_id, (r.title or "").strip(),
                        "; ".join(a.name for a in r.authors),
                        r.primary_category,
                        ",".join(sorted(r.categories)),
                        r.published.isoformat() if r.published else "",
                        r.updated.isoformat() if r.updated else "",
                        r.doi or "", r.journal_ref or "",
                        filename
                    ])

                    # Flush files periodically to make progress visible in Drive UI
                    if total % 100 == 0:
                        meta_f.flush()
                        zf.fp.flush()  # flush underlying file descriptor

                except Exception as e:
                    errors += 1
                    print(f"[{total}] ERR : {short_id} — {e}", file=sys.stderr)
                    time.sleep(1.0)

        finally:
            meta_f.close()

    print(f"\nDone. Attempted: {total} | Errors: {errors}")
    print(f"Zip archive: {zip_path}")
    print(f"Metadata CSV: {meta_path}")

if __name__ == "__main__":
    download_into_zip_drive()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[1] OK  : 2501.00663v1_Titans_Learning_to_Memorize_at_Test_Time.pdf
[2] OK  : 2501.00659v2_Why_Are_Positional_Encodings_Nonessential_for_Deep_Autoregressive_Transformers_Revisiting_a_Petroglyph.pdf
[3] OK  : 2501.00656v2_2_OLMo_2_Furious.pdf
[4] OK  : 2501.00654v3_ICONS_Influence_Consensus_for_Vision-Language_Data_Selection.pdf
[5] OK  : 2501.00644v1_Efficient_Standardization_of_Clinical_Notes_using_Large_Language_Models.pdf
[6] OK  : 2501.05464v2_LLM-MedQA_Enhancing_Medical_Question_Answering_through_Case_Studies_in_Large_Language_Models.pdf
[7] OK  : 2501.00617v1_Toward_Corpus_Size_Requirements_for_Training_and_Evaluating_Depression_Risk_Models_Using_Spoken_Language.pdf
[8] OK  : 2501.00608v1_Optimizing_Speech-Input_Length_for_Speaker-Independent_Depression_Classification.pdf
[9] OK  : 2501.00598v1__Dialogue_vs_Dialog_in_NLP_and_AI_research_Statistics_from_

[235] ERR : 2412.18351v2 — 404 Client Error: Not Found for url: https://arxiv.org/pdf/2412.18351v2


[236] OK  : 2412.18299v1_M-Ped_Multi-Prompt_Ensemble_Decoding_for_Large_Language_Models.pdf
[237] OK  : 2412.18291v2_DeepCRCEval_Revisiting_the_Evaluation_of_Code_Review_Comment_Generation.pdf
[238] OK  : 2412.18274v1_GenAI_Content_Detection_Task_2_AI_vs._Human_--_Academic_Essay_Authenticity_Challenge.pdf
[239] OK  : 2412.18260v2_Investigating_Large_Language_Models_for_Code_Vulnerability_Detection_An_Experimental_Study.pdf
[240] OK  : 2412.18216v2_ICM-Assistant_Instruction-tuning_Multimodal_Large_Language_Models_for_Rule-based_Explainable_Image_Content_Moderation.pdf
[241] OK  : 2412.18196v2_Robustness-aware_Automatic_Prompt_Optimization.pdf
[242] OK  : 2412.18194v1_VLABench_A_Large-Scale_Benchmark_for_Language-Conditioned_Robotics_Manipulation_with_Long-Horizon_Reasoning_Tasks.pdf
[243] OK  : 2412.18190v1_An_Analysis_on_Automated_Metrics_for_Evaluating_Japanese-English_Chat_Translation.pdf
[244] OK  : 2412.18188v1_On_the_Applicability_of_Zero-Shot_Cross-Lingual_Transfer_Learning_for_S

[353] ERR : 2412.16936v3 — 404 Client Error: Not Found for url: https://arxiv.org/pdf/2412.16936v3


[354] OK  : 2412.17874v2_Evaluating_LLM_Reasoning_in_the_Operations_Research_Domain_with_ORQA.pdf
[355] OK  : 2412.16933v1_Towards_a_Unified_Paradigm_Integrating_Recommendation_Systems_as_a_New_Language_in_Large_Models.pdf
[356] OK  : 2412.16926v3_Revisiting_In-Context_Learning_with_Long_Context_Language_Models.pdf
[357] OK  : 2412.16925v1_Quantifying_Public_Response_to_COVID-19_Events_Introducing_the_Community_Sentiment_and_Engagement_Index.pdf
[358] OK  : 2412.16900v1_Speech-Based_Depression_Prediction_Using_Encoder-Weight-Only_Transfer_Learning_and_a_Large_Corpus.pdf
[359] OK  : 2412.16894v1_Unsupervised_Bilingual_Lexicon_Induction_for_Low_Resource_Languages.pdf
[360] OK  : 2412.16882v2_PsychAdapter_Adapting_LLM_Transformers_to_Reflect_Traits_Personality_and_Mental_Health.pdf
[361] OK  : 2412.16877v1_Reconsidering_SMT_Over_NMT_for_Closely_Related_Languages_A_Case_Study_of_Persian-Hindi_Pair.pdf
[362] OK  : 2412.16871v1_Teaching_LLMs_to_Refine_with_Tools.pdf
[363] OK  : 2412.16855v2_